# College Query Intent Classification — 5,000-row training project

This Google Colab notebook expands your current 6-class project to **10 categories**, generates a balanced 5,000-row dataset, compares six ML models, evaluates them, selects the best model using cross-validation macro-F1, calibrates confidence scores, and creates a Streamlit deployment bundle.

**Important:** The original CSV is retained where possible. Most rows needed to reach 5,000 are synthetic template-generated examples. This is useful for building and testing the pipeline, but it does not prove real-world accuracy. Collect fresh, genuinely written and labelled student queries before making performance claims.

**Categories:** Admission, Fees, Exam, Hostel, Scholarship, Timetable, Library, Transport, Placement, Attendance.

Run every cell from top to bottom. Upload your existing project ZIP or its dataset CSV when prompted.

In [ ]:
!pip -q install pandas numpy scikit-learn joblib matplotlib seaborn streamlit


In [ ]:
# Imports and settings
import os, re, zipfile, random, json, warnings, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("/content/college_query_project")
OUT = ROOT / "artifacts"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Upload existing project ZIP OR college_query_dataset.csv
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("Upload your existing project ZIP or CSV dataset.")

uploaded_name = next(iter(uploaded))
uploaded_path = Path("/content") / uploaded_name

if uploaded_path.suffix.lower() == ".zip":
    with zipfile.ZipFile(uploaded_path) as zf:
        candidates = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if not candidates:
            raise FileNotFoundError("No CSV found inside ZIP.")
        chosen = sorted(candidates, key=lambda n: ("college_query_dataset" not in n.lower(), len(n)))[0]
        with zf.open(chosen) as f:
            raw = pd.read_csv(f)
elif uploaded_path.suffix.lower() == ".csv":
    raw = pd.read_csv(uploaded_path)
else:
    raise ValueError("Please upload a .zip or .csv file.")

print("Loaded:", uploaded_name, raw.shape, raw.columns.tolist())
display(raw.head())


In [ ]:
# Standardize the original data columns
text_col = next((c for c in ["text", "query", "question", "clean_text"] if c in raw.columns), None)
label_col = next((c for c in ["intent", "category", "label", "class"] if c in raw.columns), None)
if text_col is None or label_col is None:
    raise ValueError("Expected text/query and intent/category columns.")

real = raw[[text_col, label_col]].copy()
real.columns = ["text", "intent"]
real["text"] = real["text"].astype(str).str.strip()
real["intent"] = real["intent"].astype(str).str.strip()

label_map = {
    "admission":"Admission", "admissions":"Admission",
    "fee":"Fees", "fees":"Fees",
    "exam":"Exam", "exams":"Exam", "examination":"Exam",
    "hostel":"Hostel", "hostels":"Hostel",
    "scholarship":"Scholarship", "scholarships":"Scholarship",
    "timetable":"Timetable", "time table":"Timetable", "schedule":"Timetable",
    "library":"Library", "transport":"Transport",
    "placement":"Placement", "placements":"Placement",
    "attendance":"Attendance"
}
real["intent"] = real["intent"].str.lower().map(label_map).fillna(real["intent"])
CATEGORIES = ["Admission","Fees","Exam","Hostel","Scholarship","Timetable","Library","Transport","Placement","Attendance"]
real = real[real["intent"].isin(CATEGORIES) & (real["text"] != "")].copy()
real["source"] = "original"
real["group_id"] = [f"original_{i}" for i in range(len(real))]
print("Original labelled rows retained:", len(real))
display(real["intent"].value_counts())


## Build a balanced dataset

The following generator creates controlled synthetic student questions. It varies phrasing and slot values. Each base template has a group ID; the evaluation split keeps all variants of a template family together to reduce near-duplicate leakage.

In [ ]:
# Synthetic templates for the 10 intent categories
T = {
"Admission":[
"How can I apply for {program} admission?",
"What is the last date to submit the admission form?",
"Which documents are required for admission?",
"Can I complete the admission process {mode}?",
"Where can I check the eligibility criteria?",
"Is there an entrance test for admission?",
"How do I correct a mistake in my admission application?",
"When will the merit list be released?",
"What is the procedure for cancelling admission?",
"How can I check my admission application status?"
],
"Fees":[
"What is the total {program} fee for one semester?",
"When is the last date to pay semester fees?",
"How can I pay my college fees {mode}?",
"I paid my fees but the payment status is pending, what should I do?",
"How do I download my fee receipt?",
"Can I pay tuition fees in instalments?",
"What is the late fee after the due date?",
"How can I apply for a fee refund?",
"Where can I check my outstanding fee balance?",
"How do I report a failed fee transaction?"
],
"Exam":[
"When will the {term} semester examinations begin?",
"How can I download my examination admit card?",
"Where can I check my semester exam results?",
"What is the procedure to apply for rechecking?",
"When will the supplementary exam form be available?",
"What happens if I miss an examination?",
"How can I find the examination timetable?",
"What is the minimum passing mark in a subject?",
"How do I apply for a back-paper examination?",
"When will practical examinations take place?"
],
"Hostel":[
"How can I apply for a hostel room?",
"What is the hostel fee for {period}?",
"What documents are needed for hostel allotment?",
"Is Wi-Fi available inside the hostel?",
"What are the hostel entry and exit timings?",
"Can I change my allotted hostel room?",
"Are meals included in hostel charges?",
"How do I report a hostel maintenance problem?",
"How can I check my hostel allotment status?",
"Are AC rooms available in the hostel?"
],
"Scholarship":[
"How can I apply for a merit-based scholarship?",
"What is the eligibility for a {kind} scholarship?",
"When is the scholarship application deadline?",
"Which documents are required for scholarship verification?",
"How can I check my scholarship application status?",
"Is there a scholarship for economically weaker students?",
"Can I renew my scholarship for next semester?",
"What marks are required to qualify for a scholarship?",
"Why has my scholarship amount not been credited?",
"Where can I find the scholarship notification?"
],
"Timetable":[
"When does the {period} class start?",
"Where can I download my weekly class timetable?",
"Has the lecture schedule changed for today?",
"Are there classes on Saturday?",
"What time does the afternoon shift begin?",
"How can I check the timetable for my section?",
"When is the next practical lab session?",
"How do I report a timetable clash?",
"Has tomorrow's class been rescheduled?",
"When will the revised timetable be uploaded?"
],
"Library":[
"What are the library opening and closing hours?",
"How can I issue a book from the library?",
"How many books can a student borrow?",
"What is the book return deadline?",
"How do I renew an issued library book?",
"Is the library open on weekends?",
"How can I search the library catalogue?",
"What is the fine for returning a book late?",
"Can I access e-books and online journals?",
"How do I get a library membership card?"
],
"Transport":[
"How can I apply for the college bus service?",
"What are the available college bus routes?",
"What is the transport fee for this semester?",
"Where can I check bus pickup timings?",
"Can I change my assigned bus stop?",
"Is transport available from {place} to campus?",
"How do I get a new bus pass?",
"What should I do if the college bus is late?",
"Where can I find the transport office contact?",
"How can I renew my transport registration?"
],
"Placement":[
"How can I register for campus placement?",
"What companies are visiting for recruitment?",
"Am I eligible for placement this semester?",
"Where can I see upcoming placement drives?",
"How do I prepare for the placement aptitude test?",
"What is the minimum CGPA required for placements?",
"How can I update my resume on the placement portal?",
"When will the pre-placement talk take place?",
"How do I check my placement registration status?",
"Does the university provide career guidance?"
],
"Attendance":[
"How can I check my current attendance percentage?",
"What is the minimum attendance required for exams?",
"Why is my attendance missing for yesterday's class?",
"How do I apply for attendance correction?",
"Where can I download my attendance report?",
"Can medical leave be considered for attendance?",
"What if the teacher marked me absent by mistake?",
"How is attendance calculated for a semester?",
"How do I submit an attendance shortage application?",
"How can I check subject-wise attendance?"
]
}

SLOTS = {
"program":["BCA","MCA","MBA","BTech","computer science"],
"mode":["online","through the student portal","from the university website"],
"term":["first","second","third","final"],
"period":["this semester","next semester","morning","afternoon"],
"kind":["merit-based","need-based","sports","government"],
"place":["Kota","Gurugram","Delhi","nearby areas"]
}
def fill_slots(template):
    return re.sub(r"\{(\w+)\}", lambda m: random.choice(SLOTS.get(m.group(1), [""])), template)
def norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s']", " ", str(s).lower())).strip()

generated = []
for cat, templates in T.items():
    have = int((real.intent == cat).sum())
    need = max(0, 500 - have)
    seen = set(real.loc[real.intent == cat, "text"].map(norm))
    made, tries = 0, 0
    while made < need and tries < need * 50:
        ti = random.randrange(len(templates))
        q = fill_slots(templates[ti])
        prefix = random.choice(["", "", "Please tell me, ", "Can you tell me, ", "I want to know, "])
        suffix = random.choice(["", "", " please", " for students", " at the university"])
        q = (prefix + q[0].lower() + q[1:] + suffix).strip()
        q = q[0].upper() + q[1:]
        key = norm(q)
        tries += 1
        if len(key.split()) < 3 or key in seen:
            continue
        seen.add(key)
        generated.append({"text":q, "intent":cat, "source":"synthetic", "group_id":f"{cat}_template_{ti}"})
        made += 1
    if made < need:
        raise RuntimeError(f"Could not create enough unique examples for {cat}; add more templates.")

data = pd.concat([real[["text","intent","source","group_id"]], pd.DataFrame(generated)], ignore_index=True)
data["norm"] = data.text.map(norm)
data = data.drop_duplicates("norm").drop(columns="norm")
parts = []
for cat in CATEGORIES:
    part = data[data.intent == cat].copy()
    part["_order"] = part.source.map({"original":0,"synthetic":1})
    part = part.sort_values("_order").head(500).drop(columns="_order")
    if len(part) != 500:
        raise RuntimeError(f"{cat} has {len(part)} rows, expected 500.")
    parts.append(part)
data = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
data.to_csv(OUT / "college_query_dataset_5000.csv", index=False)

print("Final dataset:", data.shape)
display(data.groupby(["intent","source"]).size().unstack(fill_value=0))
assert len(data) == 5000 and data.intent.nunique() == 10


In [ ]:
# Dataset overview
display(data.sample(12, random_state=SEED)[["text","intent","source"]])
plt.figure(figsize=(12,5))
sns.countplot(data=data, x="intent", order=CATEGORIES)
plt.xticks(rotation=30, ha="right")
plt.title("Balanced query dataset — 500 examples per category")
plt.tight_layout()
plt.show()
print("Duplicate normalized texts:", data.text.map(norm).duplicated().sum())


In [ ]:
# Group-aware train/test split: template families cannot cross the split
X = data.text.astype(str).values
y = data.intent.astype(str).values
groups = data.group_id.astype(str).values

outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_i, test_i = next(outer.split(X, y, groups))
X_train, X_test = X[train_i], X[test_i]
y_train, y_test = y[train_i], y[test_i]
g_train = groups[train_i]
assert set(g_train).isdisjoint(set(groups[test_i]))
print("Train:", len(train_i), "Test:", len(test_i))
display(pd.DataFrame({"train":pd.Series(y_train).value_counts(), "test":pd.Series(y_test).value_counts()}))


In [ ]:
# Six ML models using a consistent TF-IDF representation
def make_pipe(clf):
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True, strip_accents="unicode",
            ngram_range=(1,2), min_df=2, max_df=0.98,
            sublinear_tf=True, max_features=50000
        )),
        ("clf", clf)
    ])

models = {
    "Multinomial Naive Bayes": make_pipe(MultinomialNB(alpha=0.5)),
    "Complement Naive Bayes": make_pipe(ComplementNB(alpha=0.5)),
    "Logistic Regression": make_pipe(LogisticRegression(max_iter=3000, C=3.0, class_weight="balanced")),
    "Linear SVM": make_pipe(LinearSVC(C=1.0, class_weight="balanced")),
    "SGD Classifier": make_pipe(SGDClassifier(loss="modified_huber", alpha=1e-5, max_iter=3000, class_weight="balanced", random_state=SEED)),
    "Random Forest": make_pipe(RandomForestClassifier(n_estimators=250, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED))
}
print(list(models))


In [ ]:
# 5-fold group-aware cross-validation on training data
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {"accuracy":"accuracy", "precision_macro":"precision_macro", "recall_macro":"recall_macro", "f1_macro":"f1_macro"}
rows = []
for name, pipe in models.items():
    print("Cross-validating:", name)
    scores = cross_validate(pipe, X_train, y_train, groups=g_train, cv=cv,
                            scoring=scoring, n_jobs=-1, error_score="raise")
    rows.append({
        "Model":name,
        "CV Accuracy":scores["test_accuracy"].mean(),
        "CV Precision (macro)":scores["test_precision_macro"].mean(),
        "CV Recall (macro)":scores["test_recall_macro"].mean(),
        "CV F1 (macro)":scores["test_f1_macro"].mean(),
        "CV Fit Time (sec)":scores["fit_time"].mean()
    })
cv_df = pd.DataFrame(rows).sort_values("CV F1 (macro)", ascending=False)
display(cv_df.style.format({c:"{:.4f}" for c in cv_df.columns if c != "Model"}))
cv_df.to_csv(OUT / "model_comparison_cv.csv", index=False)
best_name = cv_df.iloc[0]["Model"]
print("Best model chosen using CV macro-F1:", best_name)


In [ ]:
# Fit all models on training split and evaluate on the untouched holdout set
fitted, test_rows = {}, []
for name, pipe in models.items():
    print("Training:", name)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    p, r, f, _ = precision_recall_fscore_support(y_test, pred, average="macro", zero_division=0)
    test_rows.append({"Model":name, "Test Accuracy":accuracy_score(y_test,pred),
                      "Test Precision (macro)":p, "Test Recall (macro)":r, "Test F1 (macro)":f})
    fitted[name] = pipe

test_df = pd.DataFrame(test_rows).sort_values("Test F1 (macro)", ascending=False)
display(test_df.style.format({c:"{:.4f}" for c in test_df.columns if c != "Model"}))
test_df.to_csv(OUT / "model_comparison_test.csv", index=False)

# Best model is selected by CV, not by repeatedly tuning against the holdout set.
best_model = fitted[best_name]
best_pred = best_model.predict(X_test)
print(classification_report(y_test, best_pred, labels=CATEGORIES, zero_division=0))


In [ ]:
# Confusion matrix and per-class metrics for the selected model
cm = confusion_matrix(y_test, best_pred, labels=CATEGORIES)
plt.figure(figsize=(11,8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CATEGORIES, yticklabels=CATEGORIES)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — " + best_name)
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

report = classification_report(y_test, best_pred, labels=CATEGORIES, output_dict=True, zero_division=0)
pd.DataFrame(report).transpose().to_csv(OUT / "classification_report.csv")


In [ ]:
# Calibrate selected pipeline to expose predict_proba() in the Streamlit app
# Probabilities are confidence estimates, not guarantees of correctness.
calibrated = CalibratedClassifierCV(estimator=best_model, method="sigmoid", cv=3)
calibrated.fit(X_train, y_train)
joblib.dump(calibrated, OUT / "college_query_best_model.joblib")

meta = {
    "best_model_name":best_name,
    "categories":CATEGORIES,
    "dataset_rows":int(len(data)),
    "examples_per_category":500,
    "selection_metric":"5-fold StratifiedGroupKFold mean macro-F1",
    "holdout_accuracy":float(accuracy_score(y_test,best_pred)),
    "holdout_macro_f1":float(precision_recall_fscore_support(y_test,best_pred,average="macro",zero_division=0)[2]),
    "data_note":"Original examples plus synthetic template-generated examples. Benchmark metrics are not guaranteed real-world accuracy."
}
(OUT / "model_metadata.json").write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))


In [ ]:
# Create the Streamlit application
app_code = r"""
import streamlit as st
import pandas as pd
import joblib, json
from pathlib import Path

st.set_page_config(page_title="College Query AI", page_icon="🎓", layout="wide")
BASE = Path(__file__).parent

@st.cache_resource
def get_model():
    return joblib.load(BASE / "college_query_best_model.joblib")

@st.cache_data
def get_data():
    return pd.read_csv(BASE / "college_query_dataset_5000.csv")

def get_optional_csv(name):
    p = BASE / name
    return pd.read_csv(p) if p.exists() else pd.DataFrame()

model = get_model()
df = get_data()
meta_path = BASE / "model_metadata.json"
meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}

if "history" not in st.session_state:
    st.session_state.history = []

st.sidebar.title("🎓 College Query AI")
page = st.sidebar.radio("Navigation", ["Home", "Query Assistant", "Dashboard", "Model Comparison", "About"])

if page == "Home":
    st.title("🎓 College Query Intent Classification")
    st.write("NLP-based classification of student questions into college service categories.")
    a,b,c = st.columns(3)
    a.metric("Dataset rows", f"{len(df):,}")
    b.metric("Categories", df.intent.nunique())
    c.metric("Selected model", meta.get("best_model_name", "ML classifier"))
    st.subheader("Supported categories")
    st.write(" · ".join(sorted(df.intent.unique())))
    st.info("This app predicts the query category. It does not provide verified college-specific answers.")

elif page == "Query Assistant":
    st.title("🤖 Query Assistant")
    q = st.text_area("Enter your college-related question", placeholder="How can I check my attendance percentage?")
    if st.button("Classify query", type="primary"):
        if not q.strip():
            st.warning("Please enter a question.")
        else:
            pred = model.predict([q])[0]
            probs = model.predict_proba([q])[0]
            ranked = sorted(zip(model.classes_, probs), key=lambda x:x[1], reverse=True)
            conf = float(max(probs))
            st.success("Predicted category: " + pred)
            st.metric("Model confidence", f"{conf*100:.1f}%")
            if conf < 0.50:
                st.warning("Low confidence: this question may be ambiguous or outside the training examples. Verify the category.")
            p_df = pd.DataFrame(ranked, columns=["Category","Probability"])
            st.subheader("Category probabilities")
            st.bar_chart(p_df.set_index("Category"))
            st.session_state.history.insert(0, {"Query":q,"Predicted category":pred,"Confidence (%)":round(conf*100,2)})
    if st.session_state.history:
        hist = pd.DataFrame(st.session_state.history)
        st.subheader("Recent query history")
        st.dataframe(hist, use_container_width=True)
        st.download_button("Download history CSV", hist.to_csv(index=False), "query_history.csv", "text/csv")

elif page == "Dashboard":
    st.title("📊 Dataset Dashboard")
    a,b = st.columns(2)
    a.metric("Total questions", f"{len(df):,}")
    b.metric("Intent categories", df.intent.nunique())
    st.subheader("Queries per category")
    st.bar_chart(df.intent.value_counts())
    if "source" in df.columns:
        st.subheader("Original vs synthetic examples")
        st.bar_chart(df.groupby(["intent","source"]).size().unstack(fill_value=0))
    st.subheader("Dataset preview")
    st.dataframe(df[["text","intent","source"]].sample(min(25,len(df)), random_state=42), use_container_width=True)

elif page == "Model Comparison":
    st.title("🏁 Model Comparison")
    cv = get_optional_csv("model_comparison_cv.csv")
    test = get_optional_csv("model_comparison_test.csv")
    if not cv.empty:
        st.subheader("5-fold cross-validation (training split)")
        st.dataframe(cv, use_container_width=True)
        if "CV F1 (macro)" in cv:
            st.bar_chart(cv.set_index("Model")[["CV F1 (macro)"]])
    if not test.empty:
        st.subheader("Held-out test results")
        st.dataframe(test, use_container_width=True)
        if {"Test F1 (macro)","Test Accuracy"}.issubset(test.columns):
            st.bar_chart(test.set_index("Model")[["Test F1 (macro)","Test Accuracy"]])
    st.caption("Most rows in this benchmark are synthetic. Do not present these metrics as guaranteed real-world performance.")

elif page == "About":
    st.title("About the project")
    st.markdown("**Models compared:** Multinomial Naive Bayes, Complement Naive Bayes, Logistic Regression, Linear SVM, SGD Classifier and Random Forest.\\n\\n**NLP:** TF-IDF word unigrams and bigrams.\\n\\n**Categories:** Admission, Fees, Exam, Hostel, Scholarship, Timetable, Library, Transport, Placement and Attendance.\\n\\nThe classifier identifies intent only; it does not answer the underlying college-policy question.")
    st.json(meta)

st.caption("College Query AI · NLP Intent Classification")
"""
(OUT / "app.py").write_text(app_code, encoding="utf-8")
print("Created app.py")


In [ ]:
# Create requirements, README, and a deployment ZIP
(OUT / "requirements.txt").write_text("streamlit\\npandas\\nnumpy\\nscikit-learn\\njoblib\\n")
(OUT / "README.md").write_text(
    "# College Query Intent Classification\\n\\n"
    f"Selected model: {best_name}\\n\\n"
    "Run locally with: `pip install -r requirements.txt` then `streamlit run app.py`.\\n\\n"
    "Keep app.py, model, dataset, model_metadata.json and model comparison CSV files together.\\n\\n"
    "Dataset includes synthetic examples; benchmark metrics are not guaranteed real-world performance.\\n"
)

DEPLOY = ROOT / "streamlit_deployment"
DEPLOY.mkdir(exist_ok=True)
needed = [
    "app.py", "requirements.txt", "README.md",
    "college_query_best_model.joblib", "college_query_dataset_5000.csv",
    "model_comparison_cv.csv", "model_comparison_test.csv", "model_metadata.json"
]
for name in needed:
    shutil.copy2(OUT / name, DEPLOY / name)

zip_path = Path("/content/college_query_model_bundle.zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive("/content/college_query_model_bundle", "zip", DEPLOY)
print("Created:", zip_path)
from google.colab import files
files.download(str(zip_path))


## Deploy on Streamlit Community Cloud

1. Download and extract `college_query_model_bundle.zip`.
2. Upload all files inside it to your GitHub repository (same folder as `app.py`).
3. Open Streamlit Community Cloud and create a new app connected to that repository.
4. Set the main file path to `app.py`.
5. Deploy.

For your presentation, explain that the system compares six ML algorithms and selects the best using cross-validation macro-F1. Clearly disclose that the 5,000-row dataset is partly synthetic. The next serious improvement is collecting hundreds of real student queries across the 10 categories and evaluating on a completely separate real-world test set.